<a href="https://colab.research.google.com/github/VictorHugoTesti/-am-t4-s1a2026/blob/main/prova_21_Victor_Testi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline | Carregamento e Detecção de Anomalias [Fase 1 - base_ibama]

## Importar Dados do Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Bibliotecas Python

In [ ]:
!pip -q install plotly
!pip -q install yellowbrick

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

## Importar DataFrame [base_ibama] - Descrição da Base

In [ ]:
base_ibama = pd.read_csv('/content/drive/MyDrive/Aprendizagem de Maquina/prova - ML/volumeJulgamentoAI.csv', sep=';')

In [ ]:
base_ibama['Valor do Auto'] = base_ibama['Valor do Auto'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float)

In [ ]:
base_ibama['Valor do Auto'].describe()

,Valor do Auto
count,2917.000000
mean,7254.406239
std,25357.328059
min,50.000000
25%,500.000000
50%,1000.000000
75%,3500.000000
max,595500.000000


In [ ]:
base_ibama

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Número AI,Nome ou Razão Social,CPF/CNPJ,UF,Município,Status Débito,Decisão Administrativa,Tipo Auto,Tipo de Infração,Enquadramento,Data Julgamento Principal,Data Julgamento Recurso,Data Auto,Valor do Auto,Moeda,Valor Pago,Data Pagamento,Última Atualização Relatório
0,0QRNTRF2 -,RONALDO DA COSTA CUNHA,792.049.372-20,AC,CRUZEIRO DO SUL,Quitado. Baixa automática,NaN,Multa,Fauna,"Lei 9605/98 - Artigo 72, Lei 9605/98 - Artigo ...",NaN,23/02/2026,26/01/2026,500,Real,"353,5",20/02/2026,19/04/2026 21:56
1,0YIWNUKS -,JUZALDO RUFINO MOTA,411.867.022-49,AC,RIO BRANCO,Quitado. Baixa automática,NaN,Multa,Flora,Decreto 6514/2008 - Artigo 57,NaN,25/09/2025,03/04/2023,"1,000",Real,707,14/04/2023,19/04/2026 21:56
2,11175 - D,PAULO COUTO CABRAL,363.359.919-34,AC,RIO BRANCO,Quitado. Baixa automática,NaN,Multa,Flora,NaN,30/09/1998,NaN,31/08/1998,"1,240",Real,305,02/12/1998,19/04/2026 21:56
3,112481 - D,OURO BRANCO MADEIRAS IMP. E EXP. LTDA.,00.525.034/0001-08,AC,CAPIXABA,Quitado. Baixa automática,NaN,Multa,Flora,NaN,22/08/2003,NaN,20/01/2003,"5,588",Real,"3,854,3",11/09/2003,19/04/2026 21:56
4,11255 - D,SINÉZIO JOSÉ FALCÃO,139.287.402-53,AC,ACRELANDIA,Parcelado pela 1ª vez,NaN,Multa,Flora,NaN,30/09/1998,NaN,04/09/1998,500,Real,"73,4",22/03/2000,19/04/2026 21:56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2912,WIZBMOWS -,MARIA ADÉLIA FALTZ BREDA,015.205.617-33,AC,ACRELANDIA,Quitado. Baixa automática,NaN,Multa,Flora,"Lei 9605/98 - Artigo 72, Lei 9605/98 - Artigo ...",NaN,NaN,12/08/2025,"30,000",Real,"21,000",26/08/2025,19/04/2026 21:56
2913,XFSQE06B -,VICTOR FAVORETO MOURA HYPOLITTO,469.999.878-07,AC,RIO BRANCO,Quitado. Baixa automática,NaN,Multa,Fauna,"Lei 9605/98 - Artigo 72, Decreto 6514/2008 - A...",NaN,23/09/2024,14/06/2024,500,Real,"359,7",23/09/2024,19/04/2026 21:56
2914,YA6X8PV7 -,CELIO FERREIRA FURTADO,860.048.912-15,AC,SENA MADUREIRA,Quitado. Baixa automática,NaN,Multa,Fauna,"Lei 9605/98 - Artigo 70, Decreto 6514/2008 - A...",NaN,NaN,12/09/2022,"5,000",Real,"4,045,3",15/12/2023,19/04/2026 21:56
2915,ZE5A8NQQ -,FRANCISCO SERGIO FRANCO DE SOUZA,612.722.992-87,AC,BRASILEIA,Quitado. Baixa automática,NaN,Multa,Cadastro Técnico Federal,"Lei 9605/98 - Artigo 70, Decreto 6514/2008 - A...",NaN,NaN,02/09/2023,"1,000",Real,"720,8",22/11/2023,19/04/2026 21:56


# Pipeline | Visualização De Dados E Anomalias [Fase 2 - base_ibama]

## Analise exploratórias de Anomalias

In [ ]:
# Valores Nulos
base_ibama['Valor do Auto'].isnull().sum()

np.int64(0)

In [ ]:
# Valores Negativos
base_ibama.loc[base_ibama['Valor do Auto'] < 0]

,Número AI,Nome ou Razão Social,CPF/CNPJ,UF,Município,Status Débito,Decisão Administrativa,Tipo Auto,Tipo de Infração,Enquadramento,Data Julgamento Principal,Data Julgamento Recurso,Data Auto,Valor do Auto,Moeda,Valor Pago,Data Pagamento,Última Atualização Relatório


In [ ]:
# Calculando métricas
mediana = base_ibama['Valor do Auto'].median()
desvio = base_ibama['Valor do Auto'].std()

## Eixo de Alvos de Aprendizagem

In [ ]:
base_ibama.colums

In [ ]:
X_ibama = base_ibama.iloc[:, ].values

In [ ]:
X_ibama

In [ ]:
Y_ibama = base_ibama.iloc[:, ].values

In [ ]:
Y_ibama

# Pipeline | Normalização de Dados [Fase 3 - base_ibama]

# Pipeline | Divisão de Amostras de Digestores [Fase 4 - base_ibama]